# CHGNet, Mace(small, large)による計算<BR>

CHGNet, Mace によるパフォーマンスチェック（CPU/GPU 速度のみ）<BR>
様々なNNPの評価結果　https://matbench-discovery.materialsproject.org/ <BR>
  
作成：中山将伸 (2026.2.21)  再配布禁止<BR>

## 【必須】インストール　＋　ASEモジュール他のインポート


In [ ]:
!pip install ase
!pip install nglview
!pip install nglview ipywidgets==7.7.2
!pip install chgnet mace-torch

In [ ]:
import numpy as np
import math, random

import os,sys,csv,glob,shutil,re,time
from time import perf_counter
from joblib import Parallel, delayed
args = sys.argv

# sklearn
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

import ase #
from ase.optimize import LBFGS, BFGS, FIRE
from ase.constraints import FixAtoms, FixedPlane, FixBondLength
from ase.filters import UnitCellFilter, ExpCellFilter
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.verlet import VelocityVerlet
from ase.md.langevin import Langevin
from ase.md import MDLogger
from ase import Atoms
from ase.io import read, write
from ase.io import Trajectory
from ase import units
from ase.build import bulk, make_supercell
from ase.visualize import view

from google.colab import output
output.enable_custom_widget_manager()
import nglview as nv


## 【必須】Atoms object 入力

In [ ]:
#POSCAR ファイルの読み込み
#!wget
atoms=read("./Li7La3Zr2O12.cif",format="cif")

## 【必須】Atoms objectにM3GNetのポテンシャルを導入

In [ ]:
import chgnet
import torch
from chgnet.model.dynamics import CHGNetCalculator

calc = CHGNetCalculator(
    use_device="cpu",          # "cuda" も可
    on_isolated_atoms="ignore" # よく使われる設定例
)

atoms.calc = calc

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()

elapsedt=e_time-s_time
print ("computation time (chgnet_cpu)",elapsedt, "sec")




In [ ]:
from mace.calculators import mace_mp

# 軽量（small）
calc_mace_light = mace_mp(
    model="small",       # ←軽量
    device="cpu",        # "cuda" も可
    default_dtype="float32",
)

# ヘビー（large）
calc_mace_heavy = mace_mp(
    model="large",       # ←ヘビー
    device="cpu",
    default_dtype="float32",
)

# 使う方を差し替えるだけ
atoms.calc = calc_mace_light

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-light_cpu)",elapsedt, "sec")

atoms.calc = calc_mace_heavy
torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-large_cpu)",elapsedt, "sec")

In [ ]:
calc = CHGNetCalculator(
    use_device="cuda",          # "cuda" も可
    on_isolated_atoms="ignore" # よく使われる設定例
)

atoms.calc = calc

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()

elapsedt=e_time-s_time
print ("computation time (chgnet_cpu)",elapsedt, "sec")



# 軽量（small）
calc_mace_light = mace_mp(
    model="small",       # ←軽量
    device="cuda",        # "cuda" も可
    default_dtype="float32",
)

# ヘビー（large）
calc_mace_heavy = mace_mp(
    model="large",       # ←ヘビー
    device="cuda",
    default_dtype="float32",
)


atoms.calc = calc_mace_light

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-light_cpu)",elapsedt, "sec")

atoms.calc = calc_mace_heavy
torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-large_cpu)",elapsedt, "sec")


In [ ]:
from mace.calculators import mace_mp

# 軽量（small）
calc_mace_light = mace_mp(
    model="small",       # ←軽量
    device="cpu",        # "cuda" も可
    default_dtype="float32",
)

# ヘビー（large）
calc_mace_heavy = mace_mp(
    model="large",       # ←ヘビー
    device="cpu",
    default_dtype="float32",
)

# 使う方を差し替えるだけ
atoms.calc = calc_mace_light

torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-light_cpu)",elapsedt, "sec")

atoms.calc = calc_mace_heavy
torch.cuda.synchronize()
s_time = perf_counter()
energy=atoms.get_potential_energy()
force=atoms.get_forces()
torch.cuda.synchronize()
e_time = perf_counter()
elapsedt=e_time-s_time
print ("computation time(mace-large_cpu)",elapsedt, "sec")